# Visualization Gallery

The [previous EDA notebook](exploratory-data-analysis.ipynb) profiled the messy
`customers.csv` numerically. This companion notebook is a **chart-type catalogue**:
one real plot per visualization technique, against that same dataset, so you have
a reference for *which chart answers which question*. Several charts here are
**hand-drawn** with `plotters` primitives — `plotters` has no built-in box plot,
heatmap, or pair-plot, so we compute the geometry ourselves. Those are flagged
explicitly as manual implementations, not stylistic choices.

The closing [chart-selection table](#which-chart-for-which-question) summarises
when to reach for each one.

In [ ]:
:dep polars = { version = "0.44", features = ["lazy", "ndarray", "parquet", "strings"] }
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use polars::prelude::*;
use plotters::prelude::*;

let df = CsvReadOptions::default()
    .with_has_header(true)
    .try_into_reader_with_file_path(Some("/book/data/customers.csv".into()))?
    .finish()?;
println!("shape = {:?}", df.shape());
let numeric = ["age", "income", "tenure_months", "monthly_charge"];

In [ ]:
// Helper: pull a numeric column (non-null) into a Vec<f64> via the ndarray bridge.
fn col_f64(df: &DataFrame, name: &str) -> PolarsResult<Vec<f64>> {
    let m = df.clone().lazy()
        .filter(col(name).is_not_null())
        .select([col(name).cast(DataType::Float64)])
        .collect()?
        .to_ndarray::<Float64Type>(IndexOrder::C)?;
    Ok((0..m.nrows()).map(|i| m[[i, 0]]).collect())
}

// Helper: linear-interpolated quantile of a pre-sorted slice.
fn quantile(sorted: &[f64], p: f64) -> f64 {
    let n = sorted.len();
    if n == 0 { return f64::NAN; }
    let idx = (n as f64 - 1.0) * p;
    let lo = idx.floor() as usize;
    let hi = idx.ceil() as usize;
    let frac = idx - lo as f64;
    sorted[lo] * (1.0 - frac) + sorted[hi] * frac
}
println!("helpers defined");

## 1. Histogram — distribution shape

The first question about any numeric column: what's its *shape*? Histograms bin
the values and count them. A 2×2 grid, one per numeric column (income clipped
below 200k so the ~900k outlier doesn't flatten the rest):

In [ ]:
evcxr_figure((760, 540), |root| {
    root.fill(&WHITE)?;
    for (area, name) in root.split_evenly((2, 2)).iter().zip(numeric.iter()) {
        let raw = col_f64(&df, name)?;
        let vals: Vec<f64> = if *name == "income" {
            raw.into_iter().filter(|v| *v < 200_000.0).collect()
        } else { raw };
        let lo = vals.iter().cloned().fold(f64::INFINITY, f64::min);
        let hi = vals.iter().cloned().fold(f64::NEG_INFINITY, f64::max);
        let nbins = 8usize;
        let width = ((hi - lo) / nbins as f64).max(1e-9);
        let mut counts = vec![0u32; nbins];
        for v in &vals {
            let mut b = ((v - lo) / width) as usize;
            if b >= nbins { b = nbins - 1; }
            counts[b] += 1;
        }
        let maxc = *counts.iter().max().unwrap_or(&1);
        let mut chart = ChartBuilder::on(area)
            .caption(*name, ("sans-serif", 15))
            .margin(6).x_label_area_size(26).y_label_area_size(30)
            .build_cartesian_2d(lo..hi, 0u32..(maxc + 1))?;
        chart.configure_mesh().disable_mesh().draw()?;
        chart.draw_series(counts.iter().enumerate().map(|(i, &c)| {
            let x0 = lo + i as f64 * width;
            Rectangle::new([(x0, 0), (x0 + width * 0.9, c)], BLUE.filled())
        }))?;
    }
    Ok(())
})

## 2. Box plot — quartiles & outliers (hand-drawn)

`plotters` has **no box-plot primitive**, so we compute the five-number summary
(min-whisker, Q1, median, Q3, max-whisker) plus IQR outliers ourselves and draw
the box, whiskers, and outlier points from rectangles and lines. One box per
numeric column (each with its own y-scale, since their ranges differ by orders of
magnitude):

In [ ]:
evcxr_figure((760, 540), |root| {
    root.fill(&WHITE)?;
    for (area, name) in root.split_evenly((2, 2)).iter().zip(numeric.iter()) {
        let mut v = col_f64(&df, name)?;
        v.sort_by(|a, b| a.partial_cmp(b).unwrap());
        let (q1, med, q3) = (quantile(&v, 0.25), quantile(&v, 0.5), quantile(&v, 0.75));
        let iqr = q3 - q1;
        let (lof, hif) = (q1 - 1.5 * iqr, q3 + 1.5 * iqr);
        let wlo = v.iter().cloned().filter(|x| *x >= lof).fold(f64::INFINITY, f64::min);
        let whi = v.iter().cloned().filter(|x| *x <= hif).fold(f64::NEG_INFINITY, f64::max);
        let outliers: Vec<f64> = v.iter().cloned().filter(|x| *x < lof || *x > hif).collect();
        let ymin = v[0].min(wlo);
        let ymax = v[v.len() - 1].max(whi);
        let pad = (ymax - ymin) * 0.1 + 1e-9;
        let mut chart = ChartBuilder::on(area)
            .caption(*name, ("sans-serif", 15))
            .margin(6).x_label_area_size(8).y_label_area_size(46)
            .build_cartesian_2d(0.0f64..2.0f64, (ymin - pad)..(ymax + pad))?;
        chart.configure_mesh().disable_x_mesh().draw()?;
        let (xc, hw) = (1.0f64, 0.30f64);
        chart.draw_series(std::iter::once(
            Rectangle::new([(xc - hw, q1), (xc + hw, q3)], BLUE.mix(0.18).filled())))?;
        chart.draw_series(std::iter::once(
            Rectangle::new([(xc - hw, q1), (xc + hw, q3)], BLUE.stroke_width(1))))?;
        chart.draw_series(std::iter::once(
            PathElement::new(vec![(xc - hw, med), (xc + hw, med)], RED.stroke_width(2))))?;
        chart.draw_series(std::iter::once(
            PathElement::new(vec![(xc, q3), (xc, whi)], BLACK.stroke_width(1))))?;
        chart.draw_series(std::iter::once(
            PathElement::new(vec![(xc, q1), (xc, wlo)], BLACK.stroke_width(1))))?;
        chart.draw_series(std::iter::once(
            PathElement::new(vec![(xc - 0.15, whi), (xc + 0.15, whi)], BLACK.stroke_width(1))))?;
        chart.draw_series(std::iter::once(
            PathElement::new(vec![(xc - 0.15, wlo), (xc + 0.15, wlo)], BLACK.stroke_width(1))))?;
        chart.draw_series(outliers.iter().map(|&y| Circle::new((xc, y), 3, RED.filled())))?;
    }
    Ok(())
})

## 3. Correlation heatmap

How strongly does each numeric feature move with the others? We compute a Pearson
correlation matrix (rows with any missing value dropped) and render it as a
`plotters` heatmap with a **diverging** blue–white–red scale (−1 → +1) and the
value printed in each cell. This is the companion to the pair plot below — the
heatmap says *how strongly* related, the pair plot shows *what the relationship
looks like*.

In [ ]:
// Numeric matrix with null rows dropped -> Vec<Vec<f64>> (a nameable type evcxr
// can persist; the raw ndarray from `to_ndarray` can't be named across cells).
let cm: Vec<Vec<f64>> = {
    let a = df.clone().lazy()
        .select(numeric.iter().map(|c| col(*c).cast(DataType::Float64)).collect::<Vec<_>>())
        .drop_nulls(None)
        .collect()?
        .to_ndarray::<Float64Type>(IndexOrder::C)?;
    (0..a.nrows()).map(|i| (0..a.ncols()).map(|j| a[[i, j]]).collect()).collect()
};
let (rows, ncol) = (cm.len(), cm[0].len());
let mut mean = vec![0.0; ncol];
for j in 0..ncol { for i in 0..rows { mean[j] += cm[i][j]; } mean[j] /= rows as f64; }
let mut sd = vec![0.0; ncol];
for j in 0..ncol {
    let mut s = 0.0;
    for i in 0..rows { let d = cm[i][j] - mean[j]; s += d * d; }
    sd[j] = (s / rows as f64).sqrt();
}
// Precompute the correlation matrix as data (no borrowing closure -> avoids E0373).
let mut corrm = vec![vec![0.0f64; ncol]; ncol];
for a in 0..ncol { for b in 0..ncol {
    let mut c = 0.0;
    for i in 0..rows { c += (cm[i][a] - mean[a]) * (cm[i][b] - mean[b]); }
    corrm[a][b] = (c / rows as f64) / (sd[a] * sd[b]);
}}

evcxr_figure((560, 500), |root| {
    root.fill(&WHITE)?;
    let n = corrm.len();
    let mut chart = ChartBuilder::on(&root)
        .caption("Pearson correlation", ("sans-serif", 18))
        .margin(10).x_label_area_size(70).y_label_area_size(70)
        .build_cartesian_2d(0f64..n as f64, 0f64..n as f64)?;
    chart.configure_mesh().disable_mesh()
        .x_labels(n).y_labels(n)
        .x_label_formatter(&|x| numeric.get(*x as usize).map(|s| s.to_string()).unwrap_or_default())
        .y_label_formatter(&|y| numeric.get(*y as usize).map(|s| s.to_string()).unwrap_or_default())
        .draw()?;
    for a in 0..n {
        for b in 0..n {
            let r = corrm[a][b];
            let t = r.abs();
            let color = if r >= 0.0 {
                RGBColor(255, (255.0 * (1.0 - t)) as u8, (255.0 * (1.0 - t)) as u8)
            } else {
                RGBColor((255.0 * (1.0 - t)) as u8, (255.0 * (1.0 - t)) as u8, 255)
            };
            chart.draw_series(std::iter::once(
                Rectangle::new([(b as f64, a as f64), (b as f64 + 1.0, a as f64 + 1.0)], color.filled())))?;
            chart.draw_series(std::iter::once(
                Text::new(format!("{:.2}", r), (b as f64 + 0.5, a as f64 + 0.5),
                          ("sans-serif", 14).into_font().color(&BLACK))))?;
        }
    }
    Ok(())
})

## 4. Pair plot (scatter-plot matrix)

A grid of every numeric column against every other: the **diagonal** shows each
column's histogram, the **off-diagonal** panels are scatter plots (points coloured
by `churned` — orange = churned, blue = retained), so you can see the *shape* of
each pairwise relationship the heatmap above only scored.

In [ ]:
// The 4 numeric columns + churned, null rows dropped, as Vec<Vec<f64>>.
let pm: Vec<Vec<f64>> = {
    let a = df.clone().lazy()
        .select(numeric.iter().map(|c| col(*c).cast(DataType::Float64))
            .chain(std::iter::once(col("churned").cast(DataType::Float64)))
            .collect::<Vec<_>>())
        .drop_nulls(None)
        .collect()?
        .to_ndarray::<Float64Type>(IndexOrder::C)?;
    (0..a.nrows()).map(|i| (0..a.ncols()).map(|j| a[[i, j]]).collect()).collect()
};
let rows = pm.len();

evcxr_figure((820, 760), |root| {
    root.fill(&WHITE)?;
    let n = numeric.len();
    let areas = root.split_evenly((n, n));
    for r in 0..n {
        for c in 0..n {
            let area = &areas[r * n + c];
            if r == c {
                // diagonal: histogram of column r
                let vals: Vec<f64> = (0..rows).map(|i| pm[i][r]).collect();
                let lo = vals.iter().cloned().fold(f64::INFINITY, f64::min);
                let hi = vals.iter().cloned().fold(f64::NEG_INFINITY, f64::max);
                let nb = 6usize;
                let w = ((hi - lo) / nb as f64).max(1e-9);
                let mut counts = vec![0u32; nb];
                for v in &vals { let mut b = ((v - lo) / w) as usize; if b >= nb { b = nb - 1; } counts[b] += 1; }
                let maxc = *counts.iter().max().unwrap_or(&1);
                let mut ch = ChartBuilder::on(area)
                    .caption(numeric[r], ("sans-serif", 11)).margin(3)
                    .build_cartesian_2d(lo..hi, 0u32..(maxc + 1))?;
                ch.configure_mesh().disable_mesh().draw()?;
                ch.draw_series(counts.iter().enumerate().map(|(i, &cc)| {
                    let x0 = lo + i as f64 * w;
                    Rectangle::new([(x0, 0), (x0 + w * 0.9, cc)], BLUE.mix(0.5).filled())
                }))?;
            } else {
                let xs: Vec<f64> = (0..rows).map(|i| pm[i][c]).collect();
                let ys: Vec<f64> = (0..rows).map(|i| pm[i][r]).collect();
                let (xlo, xhi) = (xs.iter().cloned().fold(f64::INFINITY, f64::min),
                                  xs.iter().cloned().fold(f64::NEG_INFINITY, f64::max));
                let (ylo, yhi) = (ys.iter().cloned().fold(f64::INFINITY, f64::min),
                                  ys.iter().cloned().fold(f64::NEG_INFINITY, f64::max));
                let mut ch = ChartBuilder::on(area).margin(3)
                    .x_label_area_size(0).y_label_area_size(0)
                    .build_cartesian_2d(xlo..xhi, ylo..yhi)?;
                ch.configure_mesh().disable_mesh().draw()?;
                ch.draw_series((0..rows).map(|i| {
                    let color = if pm[i][n] > 0.5 { RGBColor(230, 140, 0) } else { RGBColor(30, 90, 200) };
                    Circle::new((pm[i][c], pm[i][r]), 2, ShapeStyle::from(color).filled())
                }))?;
            }
        }
    }
    Ok(())
})

## 5. Bar chart — categorical value counts

For a categorical column, a bar chart of value counts. We normalise `city` first
(trim + lower-case, as the [ETL chapter](../01c-etl/data-preparation.ipynb) does)
so the messy variants collapse, then count:

In [ ]:
let counts = df.clone().lazy()
    .with_columns([col("city").str().strip_chars(lit(" ")).str().to_lowercase().alias("city")])
    .group_by([col("city")])
    .agg([len().alias("n")])
    .sort(["n"], SortMultipleOptions::default().with_order_descending(true))
    .collect()?;
let cats: Vec<String> = counts.column("city")?.str()?.into_iter().map(|o| o.unwrap_or("").to_string()).collect();
let ns: Vec<f64> = col_f64(&counts, "n")?;

evcxr_figure((560, 360), |root| {
    root.fill(&WHITE)?;
    let n = cats.len();
    let maxc = ns.iter().cloned().fold(0.0, f64::max);
    let mut chart = ChartBuilder::on(&root)
        .caption("city — value counts", ("sans-serif", 16))
        .margin(10).x_label_area_size(30).y_label_area_size(35)
        .build_cartesian_2d(0f64..n as f64, 0f64..(maxc + 1.0))?;
    chart.configure_mesh().disable_x_mesh()
        .x_labels(n).x_label_formatter(&|x| cats.get(*x as usize).cloned().unwrap_or_default())
        .draw()?;
    chart.draw_series(ns.iter().enumerate().map(|(i, &c)| {
        Rectangle::new([(i as f64 + 0.1, 0.0), (i as f64 + 0.9, c)], GREEN.mix(0.7).filled())
    }))?;
    Ok(())
})

## 6. Bar chart — class balance

The same bar-chart family, applied to the target. This is the imbalance the ETL
and Evaluation chapters keep in mind — here as a chart rather than a printed
count:

In [ ]:
let bal = df.clone().lazy()
    .group_by([col("churned")]).agg([len().alias("n")])
    .sort(["churned"], SortMultipleOptions::default())
    .collect()?;
let bns: Vec<f64> = col_f64(&bal, "n")?;

evcxr_figure((420, 340), |root| {
    root.fill(&WHITE)?;
    let maxc = bns.iter().cloned().fold(0.0, f64::max);
    let mut chart = ChartBuilder::on(&root)
        .caption("class balance (churned)", ("sans-serif", 16))
        .margin(10).x_label_area_size(30).y_label_area_size(35)
        .build_cartesian_2d(0f64..2f64, 0f64..(maxc + 2.0))?;
    chart.configure_mesh().disable_x_mesh()
        .x_labels(2).x_label_formatter(&|x| if *x < 0.5 { "0 (kept)".into() } else if *x < 1.5 { "1 (churned)".into() } else { "".into() })
        .draw()?;
    chart.draw_series(bns.iter().enumerate().map(|(i, &c)| {
        let color = if i == 0 { BLUE.mix(0.7) } else { RGBColor(230, 140, 0).mix(0.9) };
        Rectangle::new([(i as f64 + 0.1, 0.0), (i as f64 + 0.9, c)], color.filled())
    }))?;
    Ok(())
})

## 7. Scatter plot with outliers flagged

The [first EDA notebook](exploratory-data-analysis.ipynb) flagged income outliers
by the IQR rule but only *printed* them. Here we render `income` vs
`monthly_charge` as a scatter, colouring IQR-flagged income points red — the
~900k customer stands out immediately on the right:

In [ ]:
let sm: Vec<Vec<f64>> = {
    let a = df.clone().lazy()
        .filter(col("income").is_not_null())
        .select([col("income").cast(DataType::Float64), col("monthly_charge").cast(DataType::Float64)])
        .collect()?
        .to_ndarray::<Float64Type>(IndexOrder::C)?;
    (0..a.nrows()).map(|i| (0..a.ncols()).map(|j| a[[i, j]]).collect()).collect()
};
let rows = sm.len();
let fence = {
    let mut inc: Vec<f64> = (0..rows).map(|i| sm[i][0]).collect();
    inc.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let (q1, q3) = (quantile(&inc, 0.25), quantile(&inc, 0.75));
    q3 + 1.5 * (q3 - q1)
};

evcxr_figure((620, 420), |root| {
    root.fill(&WHITE)?;
    let xhi = (0..rows).map(|i| sm[i][0]).fold(0.0, f64::max) * 1.05;
    let yhi = (0..rows).map(|i| sm[i][1]).fold(0.0, f64::max) * 1.10;
    let mut chart = ChartBuilder::on(&root)
        .caption("income vs monthly_charge (IQR outliers in red)", ("sans-serif", 15))
        .margin(10).x_label_area_size(36).y_label_area_size(52)
        .build_cartesian_2d(0f64..xhi, 0f64..yhi)?;
    chart.configure_mesh().x_desc("income").y_desc("monthly_charge").draw()?;
    chart.draw_series((0..rows).map(|i| {
        let flagged = sm[i][0] > fence;
        let style = if flagged { RED.filled() } else { BLUE.mix(0.6).filled() };
        Circle::new((sm[i][0], sm[i][1]), if flagged { 5 } else { 3 }, style)
    }))?;
    Ok(())
})

## 8. Missingness heatmap

Per-column null *counts* (from the first notebook) don't show whether missing
values cluster in particular rows. A rows × columns grid — one cell per value,
dark where it's missing — makes the pattern visible at a glance:

In [ ]:
let allcols = ["customer_id", "age", "income", "city", "tenure_months", "monthly_charge", "churned"];
let miss: Vec<Vec<f64>> = {
    let a = df.clone().lazy()
        .select(allcols.iter().map(|c| col(*c).is_null().cast(DataType::Float64).alias(*c)).collect::<Vec<_>>())
        .collect()?
        .to_ndarray::<Float64Type>(IndexOrder::C)?;
    (0..a.nrows()).map(|i| (0..a.ncols()).map(|j| a[[i, j]]).collect()).collect()
};
let (nrows, ncols) = (miss.len(), miss[0].len());

evcxr_figure((620, 620), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("missingness (dark = missing)", ("sans-serif", 16))
        .margin(10).x_label_area_size(80).y_label_area_size(30)
        .build_cartesian_2d(0f64..ncols as f64, 0f64..nrows as f64)?;
    chart.configure_mesh().disable_mesh()
        .x_labels(ncols).x_label_formatter(&|x| allcols.get(*x as usize).map(|s| s.to_string()).unwrap_or_default())
        .y_desc("row")
        .draw()?;
    for r in 0..nrows {
        for c in 0..ncols {
            let color = if miss[r][c] > 0.5 { RGBColor(30, 30, 30) } else { RGBColor(225, 225, 225) };
            chart.draw_series(std::iter::once(
                Rectangle::new([(c as f64, r as f64), (c as f64 + 1.0, r as f64 + 1.0)], color.filled())))?;
        }
    }
    Ok(())
})

## 9. ECDF — empirical cumulative distribution

An ECDF plots each sorted value against the fraction of data at or below it. It's
a often-skipped complement to the histogram that makes distribution shape (and
comparisons between columns) precise — no binning choice to distort it. One per
numeric column:

In [ ]:
evcxr_figure((760, 540), |root| {
    root.fill(&WHITE)?;
    for (area, name) in root.split_evenly((2, 2)).iter().zip(numeric.iter()) {
        let mut v = col_f64(&df, name)?;
        v.sort_by(|a, b| a.partial_cmp(b).unwrap());
        let n = v.len();
        let lo = v[0];
        let hi = v[n - 1];
        let pts: Vec<(f64, f64)> = v.iter().enumerate().map(|(i, &x)| (x, (i + 1) as f64 / n as f64)).collect();
        let mut chart = ChartBuilder::on(area)
            .caption(*name, ("sans-serif", 15))
            .margin(6).x_label_area_size(26).y_label_area_size(34)
            .build_cartesian_2d(lo..hi, 0f64..1.05f64)?;
        chart.configure_mesh().y_desc("F(x)").draw()?;
        chart.draw_series(LineSeries::new(pts, BLUE.stroke_width(2)))?;
    }
    Ok(())
})

## 10. Line / time chart

**No temporal column in this dataset**, so a line-over-time chart isn't
meaningful here — forcing one would be a dishonest chart. See the Time Series
chapter for this chart type applied to genuinely time-indexed data. The point
worth keeping: match the chart to the data's shape, don't manufacture structure
that isn't there.

(which-chart-for-which-question)=
## Which chart for which question

| Your question | Reach for |
| --- | --- |
| What's the shape/skew of one numeric variable? | **Histogram** or **ECDF** |
| Where are the quartiles and outliers of one variable? | **Box plot** |
| How strongly are many numeric variables related, at a glance? | **Correlation heatmap** |
| What does each pairwise relationship actually look like? | **Pair plot (scatter matrix)** |
| How frequent is each category? | **Bar chart (value counts)** |
| How balanced are the target classes? | **Bar chart (class balance)** |
| Which specific points are outliers, in context? | **Scatter with outliers flagged** |
| Do missing values cluster in particular rows/columns? | **Missingness heatmap** |
| How does a quantity evolve over an ordered axis? | **Line / time chart** (see Time Series) |

The [Model Evaluation](../01d-evaluation/cross-validation.ipynb) chapter reuses
several of these — the heatmap for confusion matrices, the bar chart for per-fold
scores — so they're worth getting comfortable with here first.